# Обучение детектора переломов в Google Colab

Этот ноутбук обучает модель **Faster R-CNN (ResNet50-FPN)** на датасете
[bone-fracture-detection-computer-vision-project](https://www.kaggle.com/datasets/pkdarabi/bone-fracture-detection-computer-vision-project).

По шагам:
1. Включаем GPU в Colab: `Runtime` → `Change runtime type` → `T4 GPU`.
2. Загружаем датасет с Kaggle через kaggle.json.
3. Запускаем обучение.
4. Скачиваем `model.pth` к себе на компьютер и кладём в `models/` локального проекта.

> **Важно про классы.** Датасет с Kaggle реально содержит 7 классов: и `humerus`, и `humerus fracture` идут отдельно. С точки зрения анатомии это одна и та же кость, поэтому мы объединяем их в один класс «плечевая кость». Поэтому в модели 6 классов переломов + фон = 7 выходов.

## 1. Проверка окружения

In [ ]:
import os

# Делает CUDA-ошибки синхронными — полезно при отладке.
# В рабочем режиме можно убрать, чтобы не терять в скорости.
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch

print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'Включи GPU в Runtime → Change runtime type → T4 GPU'

## 2. Загрузка датасета с Kaggle

Нужен файл `kaggle.json` (его можно получить в настройках профиля на kaggle.com → Account → Create New API Token).
Загрузи его в Colab через левую панель Files.

In [ ]:
!pip install -q kaggle

from pathlib import Path

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d pkdarabi/bone-fracture-detection-computer-vision-project -p /content/data --unzip
!ls /content/data

In [ ]:
# Находим корень датасета — там, где лежит data.yaml.
import yaml

DATA_ROOT = None
for p in Path('/content/data').rglob('data.yaml'):
    DATA_ROOT = p.parent
    break
print('Корень датасета:', DATA_ROOT)

with open(DATA_ROOT / 'data.yaml') as f:
    config = yaml.safe_load(f)

yolo_names = config.get('names')
# В новых ultralytics-датасетах names может быть словарём {0: 'a', 1: 'b'}.
if isinstance(yolo_names, dict):
    yolo_names = [yolo_names[i] for i in sorted(yolo_names.keys())]

print('Классы в датасете:', yolo_names)
print('Всего YOLO-классов:', len(yolo_names))

## 3. Подготовка датасета и модели

Здесь повторяем код из `src/dataset.py` и `src/model.py` локального проекта.
Так ноутбук можно запустить без клонирования репозитория.

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import v2
from torchvision import tv_tensors


# 6 анатомических областей + фон. Индекс 0 — служебный класс Faster R-CNN.
CLASS_NAMES_RU = [
    'фон',
    'локоть',
    'пальцы',
    'предплечье',
    'плечевая кость',
    'плечо',
    'запястье',
]
NUM_CLASSES = len(CLASS_NAMES_RU)

# Сопоставление имени класса из data.yaml с индексом в модели.
# 'humerus' и 'humerus fracture' оба ведут на один класс — плечевую кость.
YOLO_NAME_TO_MODEL_INDEX = {
    'elbow positive':    1,
    'fingers positive':  2,
    'forearm fracture':  3,
    'humerus fracture':  4,
    'humerus':           4,
    'shoulder fracture': 5,
    'wrist positive':    6,
}

# Строим маппинг yolo_index → model_index на основе реального data.yaml.
# Это страхует от того, что порядок классов в датасете не такой, как мы ожидали.
YOLO_TO_MODEL = {}
unknown = []
for yolo_idx, name in enumerate(yolo_names):
    key = str(name).strip().lower()
    if key in YOLO_NAME_TO_MODEL_INDEX:
        YOLO_TO_MODEL[yolo_idx] = YOLO_NAME_TO_MODEL_INDEX[key]
    else:
        unknown.append((yolo_idx, name))

print('Маппинг YOLO → модель:')
for k, v in YOLO_TO_MODEL.items():
    print(f'  {k} ({yolo_names[k]:<20}) → {v} ({CLASS_NAMES_RU[v]})')
if unknown:
    print('Неизвестные классы (будут проигнорированы):', unknown)

In [ ]:
class BoneFractureDataset(Dataset):
    """Датасет рентгеновских снимков переломов в формате YOLO."""

    IMG_EXTS = ('.jpg', '.jpeg', '.png')

    def __init__(self, root, split, transforms=None, skip_empty=True):
        self.images_dir = Path(root) / split / 'images'
        self.labels_dir = Path(root) / split / 'labels'
        self.transforms = transforms

        all_imgs = sorted(
            p for p in self.images_dir.iterdir() if p.suffix.lower() in self.IMG_EXTS
        )

        if skip_empty:
            self.image_paths = [p for p in all_imgs if self._has_valid_labels(p)]
            skipped = len(all_imgs) - len(self.image_paths)
            if skipped:
                print(f'[{split}] Пропущено {skipped} снимков без размеченных переломов.')
        else:
            self.image_paths = all_imgs

    def _label_path(self, img_path):
        return self.labels_dir / (img_path.stem + '.txt')

    def _has_valid_labels(self, img_path):
        label_path = self._label_path(img_path)
        if not label_path.exists() or label_path.stat().st_size == 0:
            return False
        try:
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                if cls not in YOLO_TO_MODEL:
                    continue
                _, _, w, h = map(float, parts[1:5])
                if w > 0 and h > 0:
                    return True
        except (ValueError, OSError):
            return False
        return False

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        w, h = image.size

        boxes, labels = [], []
        label_path = self._label_path(img_path)
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                if cls not in YOLO_TO_MODEL:
                    continue  # игнорируем неизвестные/лишние классы
                model_label = YOLO_TO_MODEL[cls]
                if not (0 < model_label < NUM_CLASSES):
                    continue  # защита от выхода за пределы

                cx, cy, bw, bh = map(float, parts[1:5])
                xmin = max(0.0, (cx - bw / 2) * w)
                ymin = max(0.0, (cy - bh / 2) * h)
                xmax = min(w - 1, (cx + bw / 2) * w)
                ymax = min(h - 1, (cy + bh / 2) * h)
                if xmax - xmin >= 1.0 and ymax - ymin >= 1.0:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(model_label)

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
class ApplyTransforms:
    """Применяет v2-преобразования к (image, target) одновременно."""

    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        boxes = tv_tensors.BoundingBoxes(
            target['boxes'],
            format=tv_tensors.BoundingBoxFormat.XYXY,
            canvas_size=(image.height, image.width),
        )
        target = dict(target)
        target['boxes'] = boxes
        image, target = self.transforms(image, target)
        target['boxes'] = target['boxes'].as_subclass(torch.Tensor)
        return image, target


train_tf = ApplyTransforms(v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
]))
eval_tf = ApplyTransforms(v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
]))

train_ds = BoneFractureDataset(DATA_ROOT, 'train', transforms=train_tf)
valid_ds = BoneFractureDataset(DATA_ROOT, 'valid', transforms=eval_tf)
print(f'Train: {len(train_ds)} образцов')
print(f'Valid: {len(valid_ds)} образцов')

In [ ]:
# Проверка одного образца: убедимся, что все labels в допустимом диапазоне
# (от 1 до NUM_CLASSES-1). Это страховка от той самой CUDA-ошибки.
img, tgt = train_ds[0]
print('Тензор:', img.shape, img.dtype, 'min/max:', float(img.min()), float(img.max()))
print('Рамки:', tgt['boxes'].shape, tgt['boxes'].dtype)
print('Метки:', tgt['labels'].tolist())
assert tgt['labels'].numel() > 0, 'Пустой target — фильтрация не сработала'
assert tgt['labels'].max().item() < NUM_CLASSES, f'Метка >= NUM_CLASSES={NUM_CLASSES}'

In [ ]:
# Гиперпараметры. Подобраны эмпирически для T4 (16 ГБ VRAM).
EPOCHS = 15
BATCH_SIZE = 4
LR = 0.005
NUM_WORKERS = 2

# persistent_workers=True — воркеры не пересоздаются между эпохами,
# это убирает безобидное, но шумное 'Exception ignored ... can only test a child process'.
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    persistent_workers=(NUM_WORKERS > 0),
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    persistent_workers=(NUM_WORKERS > 0),
)

optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 4. Обучение

In [ ]:
# Гиперпараметры. Подобраны эмпирически для T4 (16 ГБ VRAM).
EPOCHS = 15
BATCH_SIZE = 4
LR = 0.005
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total = 0.0
    pbar = tqdm(loader, desc=f'epoch {epoch} [train]')
    for images, targets in pbar:
        # На случай, если в батч случайно попадёт пустой таргет —
        # пропустим такие примеры, чтобы не падать с CUDA-ошибкой.
        keep = [i for i, t in enumerate(targets) if t['boxes'].numel() > 0]
        if not keep:
            continue
        images = [images[i].to(device) for i in keep]
        targets = [{k: v.to(device) for k, v in targets[i].items()} for i in keep]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.3f}')
    return total / len(loader)


@torch.inference_mode()
def evaluate_loss(model, loader, device):
    # Faster R-CNN считает loss только в режиме train(),
    # но мы не делаем backward — это безопасно и стандартно для валидации.
    model.train()
    total = 0.0
    for images, targets in tqdm(loader, desc='valid'):
        keep = [i for i, t in enumerate(targets) if t['boxes'].numel() > 0]
        if not keep:
            continue
        images = [images[i].to(device) for i in keep]
        targets = [{k: v.to(device) for k, v in targets[i].items()} for i in keep]
        loss_dict = model(images, targets)
        total += sum(loss_dict.values()).item()
    return total / len(loader)

In [ ]:
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    val_loss = evaluate_loss(model, valid_loader, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'[epoch {epoch}] train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), '/content/model.pth')
        print('  -> сохранил лучшие веса')

print(f'Готово. Лучший val_loss = {best_val:.4f}')

## 5. Графики обучения

In [ ]:
!pip install -q torchmetrics

from torchmetrics.detection.mean_ap import MeanAveragePrecision

test_ds = BoneFractureDataset(DATA_ROOT, 'test', transforms=eval_tf, skip_empty=False)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    persistent_workers=(NUM_WORKERS > 0),
)

model.load_state_dict(torch.load('/content/model.pth'))
model.eval()

metric = MeanAveragePrecision(iou_type='bbox')

with torch.inference_mode():
    for images, targets in tqdm(test_loader, desc='test'):
        images = [img.to(device) for img in images]
        preds = model(images)

        preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
        metric.update(preds_cpu, targets_cpu)

results = metric.compute()
print('mAP@0.5      :', float(results['map_50']))
print('mAP@0.5:0.95 :', float(results['map']))
print('mAR@100      :', float(results['mar_100']))

## 6. Оценка на тестовой выборке: mAP@0.5

mAP (mean Average Precision) — стандартная метрика качества для детекции.

In [ ]:
!pip install -q torchmetrics

from torchmetrics.detection.mean_ap import MeanAveragePrecision

test_ds = BoneFractureDataset(DATA_ROOT, 'test', transforms=eval_tf, skip_empty=False)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

model.load_state_dict(torch.load('/content/model.pth'))
model.eval()

metric = MeanAveragePrecision(iou_type='bbox')

with torch.inference_mode():
    for images, targets in tqdm(test_loader, desc='test'):
        images = [img.to(device) for img in images]
        preds = model(images)

        preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
        metric.update(preds_cpu, targets_cpu)

results = metric.compute()
print('mAP@0.5      :', float(results['map_50']))
print('mAP@0.5:0.95 :', float(results['map']))
print('mAR@100      :', float(results['mar_100']))

## 7. Скачиваем веса

Кликни ссылку — браузер скачает `model.pth`. Положи файл в `models/` локального проекта.

In [ ]:
from google.colab import files

files.download('/content/model.pth')
files.download('/content/loss_curves.png')

Готово! Теперь у тебя есть обученная модель. Положи `model.pth` в `models/` локального проекта и запусти:

```bash
docker compose up --build
```

Интерфейс откроется на http://localhost:7860.